# Liam Function-by-Function Debug Notebook

This notebook isolates each function in Liam's flow so you can run and inspect each stage independently.

Execution order:
1. Runtime preflight
2. Wrangler -> Liam CSV conversion
3. Build bulk composition (`blk_cmp`) exactly like Liam
4. `find_wet_liquidus`
5. `run_single_pressure_step`
6. `process_single_composition_parallel`
7. `parallel_melts_main_loop`
8. Workbook inspection

In [1]:
from pathlib import Path
import sys
import json
import inspect
import numpy as np
import pandas as pd

REPO_ROOT = Path('/Users/lopezama/PycharmProjects/sci-cluster')
VENDOR_DIR = REPO_ROOT / 'vendor/LeiTesting'
RUN_ROOT = REPO_ROOT / 'outputs/liam-pressure-runs'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(VENDOR_DIR) not in sys.path:
    sys.path.insert(0, str(VENDOR_DIR))

print('Repo root:', REPO_ROOT)
print('Vendor dir:', VENDOR_DIR)
print('Python executable:', sys.executable)

Repo root: /Users/lopezama/PycharmProjects/sci-cluster
Vendor dir: /Users/lopezama/PycharmProjects/sci-cluster/vendor/LeiTesting
Python executable: /Users/lopezama/miniconda3/envs/enki/bin/python


In [2]:
WRANGLED_CSV = REPO_ROOT / 'sci-data/wrangled-outputs/wrangled_KCP-109-C_compositions.csv'
SAMPLE_ID = 'KCP-109-B'

MELTS_PARAMS = {
    'Model': 'rhyolite-MELTS_v1.0.x',
    'Calculation': 'QF_P_Calc',
    'T1': 1100,
    'T2': 700,
    'ΔT': 10,
    'T unit': 'C',
    'P1': 400,
    'P2': 350,
    'ΔP': 25,
    'P unit': 'MPa',
    'fO2 offset': 0,
    'fO2 buffer': 'NNO',
    'fO2 constraint': 'TRUE',
    'ΔH': 0.5,
    'ΔV': 0,
    'ΔS': 0,
}

FIXED_OXIDE_OVERRIDES = {
    'H2O': 13.0,
    'Fe2O3': 0.18,
    'Cr2O3': 0.0,
    'NiO': 0.0,
    'CoO': 0.0,
    'CO2': 0.0,
    'SO3': 0.0,
    'Cl2O-1': 0.0,
    'F2O-1': 0.0,
}

DEBUG_PRESSURE = 125.0
VERBOSE = True

## 1) Runtime preflight

In [3]:
from sci_helpers import preflight_liam_runtime

preflight = preflight_liam_runtime(strict=False)
print(json.dumps(preflight, indent=2))

/Users/lopezama/miniconda3/envs/enki/lib/python3.10/site-packages/thermoengine/__init__.py:36: UserWarning: 

ThermoEngineLite is a work in progress. See enki-portal.org to join our developer office hours and mailing list to be kept up to date with changes.

ThermoEngineLite is not yet feature complete and may show discrepancies with the original C implementations of MELTS. A complete version is slated for release in 2026.

  from . import model


{
  "python_executable": "/Users/lopezama/miniconda3/envs/enki/bin/python",
  "python_version": "3.10.19",
  "checks": {
    "numpy": "1.26.4",
    "nptyping": "2.5.0",
    "futureproof": "missing",
    "thermoengine": "2.0.0.dev2",
    "thermoengine_path": "/Users/lopezama/miniconda3/envs/enki/lib/python3.10/site-packages/thermoengine/__init__.py"
  },
  "issues": [],
  "warnings": [
    "futureproof is missing. MeltsFP will use stdlib executor fallback. Install `futureproof==0.3.1` if you want Liam's original threading backend.",
    "Optional warning: _distutils_hack missing. This can produce noisy startup warnings in this env. Current error: No module named '_distutils_hack'"
  ]
}


## 2) Convert one wrangled sample to Liam CSV

In [4]:
from sci_helpers import build_and_write_liam_input_for_sample

liam_input_csv = RUN_ROOT / 'manual-inputs' / f'liam_input_{WRANGLED_CSV.stem}__{SAMPLE_ID}.csv'
liam_input_csv.parent.mkdir(parents=True, exist_ok=True)

liam_input_csv = build_and_write_liam_input_for_sample(
    wrangled_csv=WRANGLED_CSV,
    sample_id=SAMPLE_ID,
    melts_params=MELTS_PARAMS,
    output_csv=liam_input_csv,
    fixed_oxide_overrides=FIXED_OXIDE_OVERRIDES,
)

liam_df = pd.read_csv(liam_input_csv, index_col=0)
print('Prepared Liam CSV:', liam_input_csv)
display(liam_df.head(40))

Prepared Liam CSV: /Users/lopezama/PycharmProjects/sci-cluster/outputs/liam-pressure-runs/manual-inputs/liam_input_wrangled_KCP-109-C_compositions__KCP-109-B.csv


,KCP-109-B
SiO2,76.28822443
TiO2,0.059669847
Al2O3,13.04695036
Fe2O3,0.18
Cr2O3,0.0
FeO,0.643322296
MnO,0.035780297
MgO,0.101281661
NiO,0.0
CoO,0.0


## 3) Load Liam module and inspect function signatures

In [5]:
import MeltsFP as mf

print('MeltsFP source:', Path(mf.__file__).resolve())
print('find_wet_liquidus:', inspect.signature(mf.find_wet_liquidus))
print('run_single_pressure_step:', inspect.signature(mf.run_single_pressure_step))
print('process_single_composition_parallel:', inspect.signature(mf.process_single_composition_parallel))
print('parallel_melts_main_loop:', inspect.signature(mf.parallel_melts_main_loop))

MeltsFP source: /Users/lopezama/PycharmProjects/sci-cluster/vendor/LeiTesting/MeltsFP.py
find_wet_liquidus: (equil, T1, T2, P, n, composition, fO2_offset, verbose=False)
run_single_pressure_step: (args)
process_single_composition_parallel: (comp_data)
parallel_melts_main_loop: (compositions_csv, max_composition_workers=1, max_pressure_workers=4, verbose=True)


## 4) Build `composition`, `params`, and `blk_cmp` exactly like Liam

In [6]:
comps = pd.read_csv(liam_input_csv, index_col=0).replace(np.nan, 0, regex=True)
label = comps.columns[0]
cur_col = comps[label]

composition = pd.to_numeric(cur_col[:15]).to_dict()
params = {
    'T1': float(cur_col['T1']),
    'T2': float(cur_col['T2']),
    'delta_T': float(cur_col['ΔT']),
    'P1': float(cur_col['P1']),
    'P2': float(cur_col['P2']),
    'delta_P': float(cur_col['ΔP']),
    'const_fO2': cur_col['fO2 constraint'],
    'fO2Path': cur_col['fO2 buffer'],
    'fO2_offset': float(cur_col['fO2 offset']),
}

model_db = mf._build_model_database()
liquid = model_db.get_phase('Liq')
mol_oxides = mf.core.chem.format_mol_oxide_comp(composition, convert_grams_to_moles=True)
moles_end, oxide_res = liquid.calc_endmember_comp(
    mol_oxide_comp=mol_oxides, method='intrinsic', output_residual=True
)

if not liquid.test_endmember_comp(moles_end):
    raise RuntimeError('Infeasible endmember composition for this sample.')

mol_elm = liquid.convert_endmember_comp(moles_end, output='moles_elements')
elm_sys_main = ['H','O','Na','Mg','Al','Si','P','K','Ca','Ti','Cr','Mn','Fe','Co','Ni']
blk_cmp = np.array([mol_elm[mf.core.chem.PERIODIC_ORDER.tolist().index(elm)] for elm in elm_sys_main])

print('Label:', label)
print('composition keys:', list(composition.keys()))
print('params:', params)
print('blk_cmp shape:', blk_cmp.shape)

Label: KCP-109-B
composition keys: ['SiO2', 'TiO2', 'Al2O3', 'Fe2O3', 'Cr2O3', 'FeO', 'MnO', 'MgO', 'NiO', 'CoO', 'CaO', 'Na2O', 'K2O', 'P2O5', 'H2O']
params: {'T1': 1100.0, 'T2': 700.0, 'delta_T': 10.0, 'P1': 400.0, 'P2': 350.0, 'delta_P': 25.0, 'const_fO2': 'TRUE', 'fO2Path': 'NNO', 'fO2_offset': 0.0}
blk_cmp shape: (15,)


## 5) Run `find_wet_liquidus` directly

In [7]:
equil, _ = mf.create_equilibrate_object()
wet_liquidus_t = mf.find_wet_liquidus(
    equil=equil,
    T1=params['T1'],
    T2=params['T2'],
    P=DEBUG_PRESSURE,
    n=50,
    composition=blk_cmp,
    fO2_offset=params['fO2_offset'],
    verbose=True,
)
print('Wet liquidus temperature (C):', wet_liquidus_t)

2026-02-26 23:35:32,148 - INFO - find_wet_liquidus:465 - Computing initial state at T=900°C, P=125.0 MPa
2026-02-26 23:35:32,163 - INFO - find_wet_liquidus:517 - Computing state at T=800°C, P=125.0 MPa
2026-02-26 23:35:32,193 - INFO - find_wet_liquidus:517 - Computing state at T=750°C, P=125.0 MPa
2026-02-26 23:35:32,224 - INFO - find_wet_liquidus:517 - Computing state at T=724°C, P=125.0 MPa
2026-02-26 23:35:32,253 - INFO - find_wet_liquidus:517 - Computing state at T=712°C, P=125.0 MPa
2026-02-26 23:35:32,278 - INFO - find_wet_liquidus:517 - Computing state at T=706°C, P=125.0 MPa
2026-02-26 23:35:32,300 - INFO - find_wet_liquidus:517 - Computing state at T=702°C, P=125.0 MPa
2026-02-26 23:35:32,325 - INFO - find_wet_liquidus:517 - Computing state at T=700°C, P=125.0 MPa


Finding liquidus: T range 700.0-1100.0°C, P=125.0 MPa
Initial composition:
Liquid is the omnicomponent phase.
kc Fe3+/Fe2+ input grams Fe2O3, FeO 0.17999999999998867 0.6433222959999775
kc Fe3+/Fe2+ comp  grams Fe2O3, FeO 0.2062892403503483 0.6196669480505087
******************************** 
Calculating saturation state for Feldspar
T:1173.15 K
P:1250.0 bar
mu:[-4379534.60171922 -4415313.24889765 -4670953.29940537]
Affinity, mole fraction 13469.493854651299 [0.31173128 0.68079843 0.00747029]
 
******************************** 
Calculating saturation state for Water
T:1173.15 K
P:1250.0 bar
mu:[-413018.32500004]
Affinity, mole fraction [61667.6578538] [1.]
 
******************************** 
Calculating saturation state for Quartz
T:1173.15 K
P:1250.0 bar
mu:[-1003657.16095633]
Affinity, mole fraction [4054.65227111] [1.]
 
******************************** 
Calculating saturation state for Spinel
T:1173.15 K
P:1250.0 bar
mu:[-2227287.32355907 -2537449.04468054        0.         -1865970

## 6) Run `run_single_pressure_step` directly

In [8]:
pressure_step_args = (
    params['T1'],
    params['T2'],
    params['delta_T'],
    DEBUG_PRESSURE,
    params['const_fO2'],
    params['fO2Path'],
    params['fO2_offset'],
    0.1,
    blk_cmp,
    composition,
    label,
    VERBOSE,
)

step_result = mf.run_single_pressure_step(pressure_step_args)
print('Keys:', list(step_result.keys()))
print('Success:', step_result.get('success'))
print('num_steps:', step_result.get('num_steps'))

if step_result.get('results_data'):
    first = step_result['results_data'][0]
    print('First step keys:', list(first.keys()))
    print('First step T/P:', first['temperature'], first['pressure'])

2026-02-26 23:36:09,618 - INFO - find_wet_liquidus:465 - Computing initial state at T=900°C, P=125.0 MPa
2026-02-26 23:36:09,649 - INFO - find_wet_liquidus:517 - Computing state at T=800°C, P=125.0 MPa
2026-02-26 23:36:09,673 - INFO - find_wet_liquidus:517 - Computing state at T=750°C, P=125.0 MPa
2026-02-26 23:36:09,700 - INFO - find_wet_liquidus:517 - Computing state at T=724°C, P=125.0 MPa
2026-02-26 23:36:09,727 - INFO - find_wet_liquidus:517 - Computing state at T=712°C, P=125.0 MPa
2026-02-26 23:36:09,750 - INFO - find_wet_liquidus:517 - Computing state at T=706°C, P=125.0 MPa
2026-02-26 23:36:09,776 - INFO - find_wet_liquidus:517 - Computing state at T=702°C, P=125.0 MPa
2026-02-26 23:36:09,801 - INFO - find_wet_liquidus:517 - Computing state at T=700°C, P=125.0 MPa


Finding liquidus: T range 700.0-1100.0°C, P=125.0 MPa
Initial composition:
Liquid is the omnicomponent phase.
kc Fe3+/Fe2+ input grams Fe2O3, FeO 0.17247536842450256 0.6500930423485781
kc Fe3+/Fe2+ comp  grams Fe2O3, FeO 0.2062892403503483 0.6196669480505087
******************************** 
Calculating saturation state for Feldspar
T:1173.15 K
P:1250.0 bar
mu:[-4379534.60171922 -4415313.24889765 -4670953.29940537]
Affinity, mole fraction 13469.493854651299 [0.31173128 0.68079843 0.00747029]
 
******************************** 
Calculating saturation state for Water
T:1173.15 K
P:1250.0 bar
mu:[-413018.32500004]
Affinity, mole fraction [61667.6578538] [1.]
 
******************************** 
Calculating saturation state for Quartz
T:1173.15 K
P:1250.0 bar
mu:[-1003657.16095633]
Affinity, mole fraction [4054.65227111] [1.]
 
******************************** 
Calculating saturation state for Spinel
T:1173.15 K
P:1250.0 bar
mu:[-2227287.32355907 -2537449.04468054        0.         -1865970

## 7) Run `process_single_composition_parallel` directly

In [9]:
comp_task = (label, composition, params, blk_cmp, 1, VERBOSE)
comp_result = mf.process_single_composition_parallel(comp_task)
comp_result

2026-02-26 23:36:22,872 - INFO - find_wet_liquidus:465 - Computing initial state at T=900°C, P=400.0 MPa
2026-02-26 23:36:22,893 - INFO - find_wet_liquidus:517 - Computing state at T=800°C, P=400.0 MPa
2026-02-26 23:36:22,928 - INFO - find_wet_liquidus:517 - Computing state at T=750°C, P=400.0 MPa
2026-02-26 23:36:22,962 - INFO - find_wet_liquidus:517 - Computing state at T=724°C, P=400.0 MPa
2026-02-26 23:36:22,987 - INFO - find_wet_liquidus:517 - Computing state at T=712°C, P=400.0 MPa
2026-02-26 23:36:23,013 - INFO - find_wet_liquidus:517 - Computing state at T=706°C, P=400.0 MPa
2026-02-26 23:36:23,032 - INFO - find_wet_liquidus:517 - Computing state at T=702°C, P=400.0 MPa


Processing composition: KCP-109-B
Will process 3 pressure steps: [400. 375. 350.]
Running 3 pressure steps with futureproof (1 workers)...
Finding liquidus: T range 700.0-1100.0°C, P=400.0 MPa
Initial composition:
Liquid is the omnicomponent phase.
kc Fe3+/Fe2+ input grams Fe2O3, FeO 0.1608807094705682 0.6605260441491413
kc Fe3+/Fe2+ comp  grams Fe2O3, FeO 0.19191708150407993 0.6325991748384487
******************************** 
Calculating saturation state for Feldspar
T:1173.15 K
P:4000.0 bar
mu:[-4349602.33119639 -4383356.76925314 -4642645.21831539]
Affinity, mole fraction 12087.532808286624 [0.30077559 0.69209579 0.00712862]
 
******************************** 
Calculating saturation state for Water
T:1173.15 K
P:4000.0 bar
mu:[-407223.84265443]
Affinity, mole fraction [67757.19706898] [1.]
 
******************************** 
Calculating saturation state for Quartz
T:1173.15 K
P:4000.0 bar
mu:[-996439.18590635]
Affinity, mole fraction [3332.9999136] [1.]
 
***************************

2026-02-26 23:36:23,056 - INFO - find_wet_liquidus:517 - Computing state at T=700°C, P=400.0 MPa
2026-02-26 23:36:23,130 - INFO - find_wet_liquidus:465 - Computing initial state at T=900°C, P=375.0 MPa
2026-02-26 23:36:23,152 - INFO - find_wet_liquidus:517 - Computing state at T=800°C, P=375.0 MPa
2026-02-26 23:36:23,179 - INFO - find_wet_liquidus:517 - Computing state at T=750°C, P=375.0 MPa
2026-02-26 23:36:23,203 - INFO - find_wet_liquidus:517 - Computing state at T=724°C, P=375.0 MPa
2026-02-26 23:36:23,228 - INFO - find_wet_liquidus:517 - Computing state at T=712°C, P=375.0 MPa
2026-02-26 23:36:23,260 - INFO - find_wet_liquidus:517 - Computing state at T=706°C, P=375.0 MPa


Liquid is the omnicomponent phase.
kc Fe3+/Fe2+ input grams Fe2O3, FeO 0.2616691850566307 0.5698354639912315
kc Fe3+/Fe2+ comp  grams Fe2O3, FeO 0.2627198234532752 0.5688900879928963
******************************** 
Calculating saturation state for Feldspar
T:973.15 K
P:4000.0 bar
mu:[-4223087.69670499 -4257408.25166927 -4512913.08267166]
Affinity, mole fraction 865.8956056752099 [0.24571104 0.74981828 0.00447068]
 
******************************** 
Calculating saturation state for Water
T:973.15 K
P:4000.0 bar
mu:[-377381.31536023]
Affinity, mole fraction [68934.57918126] [1.]
 
******************************** 
Calculating saturation state for Quartz
T:973.15 K
P:4000.0 bar
mu:[-969686.68780283]
Affinity, mole fraction [795.14665305] [1.]
 
******************************** 
Calculating saturation state for Spinel
T:973.15 K
P:4000.0 bar
mu:[-2134653.0060279  -2456021.36368394        0.         -1745146.23933771
 -1335520.50954251]
Affinity, mole fraction 1892.1049419352753 [-0.00221

2026-02-26 23:36:23,284 - INFO - find_wet_liquidus:517 - Computing state at T=702°C, P=375.0 MPa
2026-02-26 23:36:23,328 - INFO - find_wet_liquidus:517 - Computing state at T=700°C, P=375.0 MPa
2026-02-26 23:36:23,411 - INFO - find_wet_liquidus:465 - Computing initial state at T=900°C, P=350.0 MPa
2026-02-26 23:36:23,432 - INFO - find_wet_liquidus:517 - Computing state at T=800°C, P=350.0 MPa
2026-02-26 23:36:23,456 - INFO - find_wet_liquidus:517 - Computing state at T=750°C, P=350.0 MPa
2026-02-26 23:36:23,481 - INFO - find_wet_liquidus:517 - Computing state at T=724°C, P=350.0 MPa


Liquid is the omnicomponent phase.
kc Fe3+/Fe2+ input grams Fe2O3, FeO 0.25821607135280067 0.5729426137256689
kc Fe3+/Fe2+ comp  grams Fe2O3, FeO 0.26127696368741254 0.5701883890981735
******************************** 
Calculating saturation state for Feldspar
T:979.15 K
P:3750.0 bar
mu:[-4229352.16129999 -4263832.60358823 -4519095.14938025]
Affinity, mole fraction 1275.984493381159 [0.24906189 0.746326   0.00461211]
 
******************************** 
Calculating saturation state for Water
T:979.15 K
P:3750.0 bar
mu:[-378660.10744624]
Affinity, mole fraction [68656.87734063] [1.]
 
******************************** 
Calculating saturation state for Quartz
T:979.15 K
P:3750.0 bar
mu:[-971095.66515016]
Affinity, mole fraction [927.03908595] [1.]
 
******************************** 
Calculating saturation state for Spinel
T:979.15 K
P:3750.0 bar
mu:[-2138045.64845126 -2459034.80114825        0.         -1749292.61973893
 -1339467.55484068]
Affinity, mole fraction 2369.3664814915546 [-0.002

2026-02-26 23:36:23,508 - INFO - find_wet_liquidus:517 - Computing state at T=712°C, P=350.0 MPa
2026-02-26 23:36:23,546 - INFO - find_wet_liquidus:517 - Computing state at T=706°C, P=350.0 MPa
2026-02-26 23:36:23,561 - INFO - find_wet_liquidus:517 - Computing state at T=702°C, P=350.0 MPa
2026-02-26 23:36:23,591 - INFO - find_wet_liquidus:517 - Computing state at T=700°C, P=350.0 MPa


Liquid is the omnicomponent phase.
kc Fe3+/Fe2+ input grams Fe2O3, FeO 0.25397444078516807 0.5767592796168566
kc Fe3+/Fe2+ comp  grams Fe2O3, FeO 0.2598954819734779 0.5714314615564655
******************************** 
Calculating saturation state for Feldspar
T:985.15 K
P:3500.0 bar
mu:[-4235636.75436679 -4270277.09811634 -4525298.663185  ]
Affinity, mole fraction 1691.7488771047665 [0.25237815 0.74286855 0.0047533 ]
 
******************************** 
Calculating saturation state for Water
T:985.15 K
P:3500.0 bar
mu:[-379951.07149676]
Affinity, mole fraction [68363.74879909] [1.]
 
******************************** 
Calculating saturation state for Quartz
T:985.15 K
P:3500.0 bar
mu:[-972509.03606354]
Affinity, mole fraction [1060.59896076] [1.]
 
******************************** 
Calculating saturation state for Spinel
T:985.15 K
P:3500.0 bar
mu:[-2141451.61700781 -2462061.6711537         0.         -1753453.50748773
 -1343431.67421379]
Affinity, mole fraction 2854.326676401583 [-0.003

{'label': 'KCP-109-B',
 'filename': 'parallel-results/fp-test/Parallel_MELTS_KCP-109-B_dP25.0_02-26_1cores.xlsx',
 'num_pressure_steps': 3,
 'num_data_points': 3,
 'P_QF': None,
 'P_Q2F': None,
 'total_time': 0.9356000423431396,
 'calc_time': 0.054181575775146484,
 'pressure_error': "Worksheet named 'quartz' not found"}

## 8) Run full `parallel_melts_main_loop` on the prepared Liam CSV

In [ ]:
loop_results = mf.parallel_melts_main_loop(
    str(liam_input_csv),
    max_composition_workers=1,
    max_pressure_workers=1,
    verbose=VERBOSE,
)
loop_results

## 9) Inspect workbook output from the full loop

In [ ]:
from openpyxl import load_workbook

if loop_results and 'filename' in loop_results[0]:
    wb_path = Path(loop_results[0]['filename']).expanduser().resolve()
    print('Workbook:', wb_path)
    wb = load_workbook(wb_path)
    print('Sheets:', wb.sheetnames)
    print('Rows per sheet:', {s: wb[s].max_row for s in wb.sheetnames})
else:
    print('No workbook filename found in loop result.')

## 10) Optional: run via wrapper API in `sci_helpers`

In [ ]:
from sci_helpers import PreparedInputRunConfig, run_single_liam_pressure_from_prepared_csv

wrapper_cfg = PreparedInputRunConfig(
    prepared_liam_csv=liam_input_csv,
    max_composition_workers=1,
    max_pressure_workers=1,
    vendor_code_dir=VENDOR_DIR,
    run_root_dir=RUN_ROOT,
    verbose=False,
)

wrapper_summary = run_single_liam_pressure_from_prepared_csv(wrapper_cfg)
print(json.dumps({k: v for k, v in wrapper_summary.items() if k not in {'results', 'preflight'}}, indent=2))
display(pd.DataFrame(wrapper_summary['results']))